# Diffusion Models Math

Part of **ML Math Applied**.

## 1. Intuition First

A diffusion model destroys structure gradually with Gaussian noise, then learns to reverse that corruption one step at a time.

![Diffusion process](../assets/diagrams/diffusion_forward_reverse.svg)

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
np.set_printoptions(precision=5, suppress=True)
torch.set_printoptions(precision=5)

## 2. Forward Process Derivation

With schedule $\{\beta_t\}_{t=1}^T$, define $\alpha_t = 1 - \beta_t$ and $\bar{\alpha}_t = \prod_{s=1}^t \alpha_s$.
Repeatedly applying

$$
q(x_t \mid x_{t-1}) = \mathcal{N}(\sqrt{\alpha_t} x_{t-1}, (1-\alpha_t) I)
$$

yields the closed form

$$
q(x_t \mid x_0) = \mathcal{N}(\sqrt{\bar{\alpha}_t} x_0, (1-\bar{\alpha}_t) I).
$$

So one noisy sample can be written directly as

$$
x_t = \sqrt{\bar{\alpha}_t} x_0 + \sqrt{1-\bar{\alpha}_t} \, \epsilon,
\qquad
\epsilon \sim \mathcal{N}(0, I).
$$

In [ ]:
from src.ml_components import diffusion_forward_process, linear_beta_schedule, predict_clean_from_noise

betas = linear_beta_schedule(30, beta_start=1e-4, beta_end=0.04)
clean_point = np.array([[1.2, -0.4]])
known_noise = np.array([[0.3, -0.2]])
noisy_point, cache = diffusion_forward_process(clean_point, timestep=18, betas=betas, noise=known_noise)
recovered_point = predict_clean_from_noise(noisy_point, timestep=18, predicted_noise=known_noise, betas=betas)

rng = np.random.default_rng(7)
cluster = np.concatenate(
    [rng.normal(loc=(-1.0, -0.6), scale=0.18, size=(60, 2)), rng.normal(loc=(1.0, 0.8), scale=0.18, size=(60, 2))],
    axis=0,
)
noisy_snapshots = {}
for timestep in (0, 10, 20, 29):
    snapshot_noise = rng.standard_normal(cluster.shape)
    noisy_snapshots[timestep], _ = diffusion_forward_process(cluster, timestep=timestep, betas=betas, noise=snapshot_noise)

print("noisy point =", noisy_point)
print("recovered point =", recovered_point)

## 3. PyTorch Verification

The closed-form reconstruction of `x_0` from `x_t` and the true noise should match exactly.

In [ ]:
betas_t = torch.tensor(betas, dtype=torch.float64)
alpha_bar_t = torch.cumprod(1.0 - betas_t, dim=0)[18]
clean_t = torch.tensor(clean_point, dtype=torch.float64)
noise_t = torch.tensor(known_noise, dtype=torch.float64)
noisy_t = torch.sqrt(alpha_bar_t) * clean_t + torch.sqrt(1.0 - alpha_bar_t) * noise_t
recovered_t = (noisy_t - torch.sqrt(1.0 - alpha_bar_t) * noise_t) / torch.sqrt(alpha_bar_t)

print(torch.allclose(noisy_t, torch.tensor(noisy_point), atol=1e-8))
print(torch.allclose(recovered_t, torch.tensor(recovered_point), atol=1e-8))

## 4. Custom Figures

The first panel tracks the signal and noise coefficients; the second row shows a 2D dataset being diffused.

In [ ]:
alpha_bar = np.cumprod(1.0 - betas)
signal = np.sqrt(alpha_bar)
noise = np.sqrt(1.0 - alpha_bar)

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
axes[0, 0].plot(signal, label="sqrt(alpha_bar_t)", linewidth=2.5)
axes[0, 0].plot(noise, label="sqrt(1 - alpha_bar_t)", linewidth=2.5)
axes[0, 0].set_title("Signal-to-noise coefficients")
axes[0, 0].legend()

axes[0, 1].axis("off")
axes[0, 1].text(0.05, 0.75, "As t grows:", fontsize=14)
axes[0, 1].text(0.05, 0.55, "- signal coefficient shrinks", fontsize=12)
axes[0, 1].text(0.05, 0.40, "- noise coefficient grows", fontsize=12)
axes[0, 1].text(0.05, 0.25, "- data becomes close to Gaussian", fontsize=12)

for axis, timestep in zip(axes[1], (0, 29)):
    points = noisy_snapshots[timestep]
    axis.scatter(points[:, 0], points[:, 1], alpha=0.65, s=18)
    axis.set_title(f"cluster at timestep {timestep}")
    axis.set_xlabel("x1")
    axis.set_ylabel("x2")

plt.tight_layout()
plt.show()

## 5. Case Study: Why Predict Noise

In image diffusion models, predicting the residual noise keeps the target distribution stable across time.
That makes the denoising network easier to train than directly predicting a fully clean sample at every step.